# Paper 74: The Evolution of the Solar Wind (Vidotto 2021) / 태양풍의 진화

**Reference**: Vidotto, A. A., *Living Reviews in Solar Physics*, 18:3, 2021. DOI: 10.1007/s41116-021-00029-w

**English.** This notebook reproduces the key quantitative tools of Vidotto's (2021) Living Review, placing the present-day Sun within an evolutionary sequence of solar-like stellar winds. We (1) solve Parker's isothermal and polytropic winds, (2) plot the empirical mass-loss-versus-age sequence, (3) compute the Alfvén radius and angular-momentum loss rate, (4) derive and visualise the Skumanich spin-down law from magnetic braking, (5) simulate a toy Ly-alpha absorption profile from an astrospheric hydrogen wall, (6) estimate atmospheric erosion of Earth/Mars from the young solar wind, (7) plot the empirical Mdot-Fx scaling, and (8) trace heliospheric radius evolution.

**한국어.** 이 노트북은 Vidotto(2021) Living Review의 핵심 정량 도구들을 재현하여 현재 태양을 태양형 항성풍의 진화 시퀀스 안에 배치한다. (1) Parker 등온·폴리트로픽 항성풍을 풀고, (2) 질량 손실률-연령 경험 시퀀스를 도시하며, (3) Alfvén 반경과 각운동량 손실률을 계산하고, (4) 자기 제동에서 Skumanich 감속 법칙을 유도·시각화하며, (5) astrosphere 수소 벽의 Ly-alpha 흡수 프로파일을 토이 모델로 시뮬레이션하고, (6) 젊은 태양풍에 의한 지구·화성 대기 침식을 추정하고, (7) 경험적 Mdot-Fx scaling을 도시하고, (8) 태양권 반경 진화를 추적한다.

## Setup / 설정

**English.** Import numerical/plotting libraries and define CGS physical constants used throughout.

**한국어.** 수치·플로팅 라이브러리를 가져오고 노트북 전체에서 사용할 cgs 물리 상수를 정의한다.

In [ ]:
"""Global imports and physical constants in cgs units.

All numerical constants follow standard cgs conventions:
length in cm, mass in g, time in s, field strength in gauss.
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import odeint

# Fundamental constants (cgs).
G = 6.674e-8          # cm^3 g^-1 s^-2
kB = 1.381e-16        # erg K^-1
mp = 1.673e-24        # g (proton mass)
c_light = 2.998e10    # cm s^-1

# Solar values.
M_sun = 1.989e33              # g
R_sun = 6.957e10              # cm
L_sun = 3.828e33              # erg s^-1
Omega_sun = 2.7e-6            # rad s^-1 (sidereal)
Mdot_sun = 2.0e-14            # M_sun yr^-1 (Vidotto 2021 adopted value)
u_inf_sun = 4.0e7             # cm s^-1 (~400 km/s)
age_sun_gyr = 4.6             # Gyr
au = 1.496e13                 # cm
yr = 3.156e7                  # s

# Mean molecular weight for fully ionised H plasma (0.5*mp).
mu = 0.5

print(f"Sun: Mdot = {Mdot_sun:.1e} M_sun/yr = {Mdot_sun*M_sun/yr:.2e} g/s")
print(f"Sun: Omega = {Omega_sun:.2e} rad/s  (P_rot = {2*np.pi/Omega_sun/86400:.1f} d)")

## 1. Parker Isothermal and Polytropic Wind Solutions / Parker 등온·폴리트로픽 해

**English.** The polytropic momentum equation (Vidotto 2021, Eq. 42) is:
$$\frac{1}{u_r}\frac{du_r}{dr} = \frac{-GM_\star/r^2 + 2c_s^2/r}{u_r^2 - c_s^2}$$
The only physical solution is the transonic one through the critical point $r_c = GM_\star/(2c_s^2)$. For the isothermal case ($\Gamma = 1$), $c_s = a$ is constant; for polytropic ($\Gamma < 5/3$), $c_s^2 = \Gamma a^2$ varies with $\rho$.

**한국어.** 폴리트로픽 운동량 방정식(Vidotto 2021, 식 42): 위 식. 물리적 해는 임계점 $r_c = GM_\star/(2c_s^2)$을 지나는 transonic 해. 등온($\Gamma = 1$)은 $c_s = a$ 상수; 폴리트로픽($\Gamma < 5/3$)은 $c_s^2 = \Gamma a^2$가 $\rho$에 따라 변함.

In [ ]:
def parker_isothermal(r_over_rc, cs):
    """Return transonic Parker isothermal wind velocity u(r).

    Args:
        r_over_rc: Array of radii in units of the critical radius r_c.
        cs: Isothermal sound speed in cm/s.

    Returns:
        Array of wind velocity u in cm/s.
    """
    # Parker equation in dimensionless form:
    # (u/cs)^2 - ln((u/cs)^2) = 4 ln(r/rc) + 4 rc/r - 3
    u = np.zeros_like(r_over_rc)
    for i, x in enumerate(r_over_rc):
        rhs = 4 * np.log(x) + 4 / x - 3
        if x < 1.0:
            # Subsonic branch: u/cs < 1
            f = lambda v: v**2 - np.log(v**2) - rhs
            u[i] = cs * brentq(f, 1e-3, 0.999)
        else:
            f = lambda v: v**2 - np.log(v**2) - rhs
            u[i] = cs * brentq(f, 1.001, 20.0)
    return u

# Compute for several coronal temperatures.
temperatures = [0.75e6, 1.5e6, 3.0e6, 4.0e6]
colors = ['cyan', 'green', 'black', 'magenta']

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
r_over_rc = np.logspace(-1.5, 2.5, 200)
for T, col in zip(temperatures, colors):
    cs = np.sqrt(kB * T / (mu * mp))
    rc = G * M_sun / (2 * cs**2)
    u = parker_isothermal(r_over_rc, cs)
    r_stellar = r_over_rc * rc / R_sun
    label = f'T = {T/1e6:.2f} MK, r_c = {rc/R_sun:.1f} R_sun'
    ax[0].plot(r_stellar, u/1e5, color=col, label=label)
    ax[1].plot(r_stellar[r_stellar < 10], u[r_stellar < 10]/1e5, color=col, label=label)

for a in ax:
    a.set_xlabel('r / R_sun')
    a.set_ylabel('wind speed (km/s)')
    a.grid(alpha=0.3)
    a.legend(loc='best', fontsize=8)
ax[0].set_xscale('log')
ax[0].set_title('Parker isothermal wind (full range)')
ax[1].set_title('Zoom-in: sonic points (filled circles)')
for T, col in zip(temperatures, colors):
    cs = np.sqrt(kB * T / (mu * mp))
    rc_r = G * M_sun / (2 * cs**2) / R_sun
    ax[1].plot(rc_r, cs/1e5, 'o', color=col, ms=8)
plt.tight_layout()
plt.show()

**Observation / 관측.** The sonic point $r_c$ moves inward (~5 R_sun) at high T (4 MK) and outward (~200 R_sun) at low T (0.75 MK). Asymptotic speeds span 400-1200 km/s. This matches Vidotto's Fig. 22. / 음속점 $r_c$는 고온(4 MK)에서 ~5 R_sun, 저온(0.75 MK)에서 ~200 R_sun. 종단 속도는 400-1200 km/s 범위. Vidotto Fig. 22와 일치.

## 2. Mass-loss Rate Evolution with Age / 연령에 따른 질량 손실률 진화

**English.** Vidotto (2021) derives two evolutionary power-laws: the gentler $\dot M \propto t^{-0.99}$ (using Güdel 2007 $L_X$-age relation, her Eq. 18) and the steeper $\dot M \propto t^{-2.33}$ from Wood et al. (2005a). Both normalised to $\dot M_\odot = 2\times 10^{-14}\,M_\odot$/yr at 4.6 Gyr. We also show the Johnstone (2015a) saturation at ~$100\,\dot M_\odot$ for young ages, and the V830 Tau upper limit at 2 Myr.

**한국어.** Vidotto(2021)는 두 진화 멱법칙을 도출한다: 약한 경사 $\dot M \propto t^{-0.99}$(Güdel 2007 $L_X$-연령 관계, 식 18)와 강한 경사 $\dot M \propto t^{-2.33}$(Wood et al. 2005a). 둘 다 4.6 Gyr에서 $\dot M_\odot = 2\times 10^{-14}\,M_\odot$/yr로 정규화. Johnstone(2015a) 포화(~$100\,\dot M_\odot$)와 V830 Tau 상한(2 Myr)도 함께 도시.

In [ ]:
def mdot_vidotto(age_gyr, exponent=-0.99):
    """Return mass-loss rate in solar units using Vidotto 2021 power-law.

    Args:
        age_gyr: Age in Gyr.
        exponent: Power-law index. Default -0.99 (Güdel 2007 based).

    Returns:
        Mass-loss rate in units of present-day solar Mdot.
    """
    return (age_gyr / age_sun_gyr) ** exponent

ages = np.logspace(-3, 1, 500)
mdot_shallow = mdot_vidotto(ages, -0.99)
mdot_steep = mdot_vidotto(ages, -2.33)

mdot_saturated = np.minimum(mdot_shallow, 100.0)

fig, ax = plt.subplots(figsize=(9, 6))
ax.loglog(ages, mdot_shallow * Mdot_sun, 'b-',
          label=r'$\dot M \propto t^{-0.99}$ (Vidotto 2021 Eq. 18)', lw=2)
ax.loglog(ages, mdot_steep * Mdot_sun, 'r--',
          label=r'$\dot M \propto t^{-2.33}$ (Wood et al. 2005a)', lw=2)
ax.axhline(100 * Mdot_sun, color='gray', linestyle=':',
           label='Saturation (Johnstone 2015a, ~100 Mdot_sun)')

ax.plot(0.002, 3e-9, 'kv', ms=12, label='V830 Tau upper limit (2 Myr)')
ax.plot(age_sun_gyr, Mdot_sun, 'y*', ms=20, label='Present-day Sun')

ax.set_xlabel('Stellar age (Gyr)')
ax.set_ylabel(r'$\dot M$ (M_sun / yr)')
ax.set_title('Solar wind mass-loss rate evolution (Vidotto 2021, Fig. 11)')
ax.grid(alpha=0.3, which='both')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

print('Young-Sun mass-loss estimates at 0.1 Gyr:')
print(f'  Shallow (t^-0.99):  {mdot_vidotto(0.1, -0.99):.1f} x Mdot_sun')
print(f'  Steep (t^-2.33):    {mdot_vidotto(0.1, -2.33):.1f} x Mdot_sun')

## 3. Alfvén Radius and Angular Momentum Loss / Alfvén 반경과 각운동량 손실

**English.** The Weber-Davis (1967) angular-momentum loss rate (Vidotto 2021, Eq. 62) is
$$\dot J = \tfrac{2}{3}\,\dot M\,\Omega_\star\, r_A^2$$
The Alfvén radius $r_A$ is where $u_r = v_A = B/\sqrt{4\pi\rho}$. For a dipolar field, $r_A \sim 10 R_\star (B_\star/1\,\textrm{G})^{2/7}(\dot M/\dot M_\odot)^{-2/7}$ (Kawaler 1988 with $n = 3/7$).

**한국어.** Weber-Davis(1967) 각운동량 손실률(Vidotto 2021, 식 62)은 위 식. Alfvén 반경 $r_A$는 $u_r = v_A = B/\sqrt{4\pi\rho}$인 지점. 쌍극자장: $r_A \sim 10 R_\star (B_\star/1\,\textrm{G})^{2/7}(\dot M/\dot M_\odot)^{-2/7}$(Kawaler 1988, $n = 3/7$).

In [ ]:
def alfven_radius(B_star_gauss, mdot_ratio=1.0, n_exponent=3/7):
    """Estimate Alfvén radius for a dipolar geometry.

    Args:
        B_star_gauss: Surface magnetic field strength in gauss.
        mdot_ratio: Mass-loss rate in units of solar.
        n_exponent: Kawaler 1988 geometry exponent (3/7 = dipolar).

    Returns:
        Alfvén radius in stellar radii.
    """
    rA_sun = 12.0
    B_sun = 1.0  # characteristic solar large-scale field in gauss
    return rA_sun * (B_star_gauss / B_sun) ** (2 * n_exponent) * mdot_ratio ** (-2 * n_exponent)

def jdot(mdot_ratio, omega_ratio, rA_stellar_radii):
    """Compute angular momentum loss rate in cgs erg.

    Args:
        mdot_ratio: Mass-loss rate in units of solar.
        omega_ratio: Rotation rate in units of solar.
        rA_stellar_radii: Alfvén radius in stellar radii.

    Returns:
        Angular momentum loss rate in erg.
    """
    return (2.0/3.0) * mdot_ratio * Mdot_sun * M_sun / yr * \
           omega_ratio * Omega_sun * (rA_stellar_radii * R_sun) ** 2

ages_fine = np.linspace(0.1, 10, 200)
omega_ratio = (age_sun_gyr / ages_fine) ** 0.5  # Skumanich
B_ratio = omega_ratio  # linear dynamo
mdot_ratio_arr = mdot_vidotto(ages_fine, -0.99)
rA_arr = np.array([alfven_radius(B_ratio[i], mdot_ratio_arr[i]) for i in range(len(ages_fine))])
jdot_arr = np.array([jdot(mdot_ratio_arr[i], omega_ratio[i], rA_arr[i])
                     for i in range(len(ages_fine))])

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].loglog(ages_fine, rA_arr, 'b-')
ax[0].axvline(age_sun_gyr, color='gray', linestyle=':', label='Present Sun')
ax[0].set_xlabel('Age (Gyr)'); ax[0].set_ylabel(r'$r_A / R_\star$')
ax[0].set_title('Alfvén radius evolution'); ax[0].grid(alpha=0.3); ax[0].legend()
ax[1].loglog(ages_fine, jdot_arr, 'r-')
ax[1].axvline(age_sun_gyr, color='gray', linestyle=':')
ax[1].set_xlabel('Age (Gyr)'); ax[1].set_ylabel(r'$\dot J$ (erg)')
ax[1].set_title('Angular momentum loss rate'); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Present-day Sun: r_A ~ {alfven_radius(1.0, 1.0):.1f} R_sun')
print(f'Present-day Sun: Jdot ~ {jdot(1.0, 1.0, 12.0):.2e} erg')

## 4. Skumanich Spin-down Law / Skumanich 감속 법칙

**English.** From Vidotto (2021) Sect. 5.3.1, integrating the Weber-Davis loss formula with $B_\star \propto \Omega_\star$ (linear dynamo) yields $\Omega_\star \propto t^{-1/2}$. We integrate $d\Omega/dt \propto -\Omega^3$ numerically for a toy model and compare with the analytic $t^{-1/2}$ curve and observed open-cluster rotation percentiles.

**한국어.** Vidotto(2021) 5.3.1절에 따라, $B_\star \propto \Omega_\star$(선형 다이나모)를 가정하고 Weber-Davis 손실 공식을 적분하면 $\Omega_\star \propto t^{-1/2}$가 얻어진다. 토이 모델로 $d\Omega/dt \propto -\Omega^3$를 수치 적분하고 분석 해 $t^{-1/2}$ 및 관측된 개방 성단 자전 percentile과 비교.

In [ ]:
def domega_dt(omega, t, A):
    """Toy rotational-evolution ODE following Kawaler 1988.

    Args:
        omega: Current rotation rate in rad/s.
        t: Time in s (not used explicitly since ODE is autonomous).
        A: Braking constant calibrated so omega(4.6 Gyr) = Omega_sun.

    Returns:
        d omega / d t in rad/s^2.
    """
    return -A * omega ** 3

t0_s = 0.05 * 1e9 * yr
tf_s = 4.6 * 1e9 * yr
Omega_0 = 10.0 * Omega_sun
A = (Omega_sun**-2 - Omega_0**-2) / (2 * (tf_s - t0_s))

t_arr_s = np.logspace(np.log10(t0_s), np.log10(tf_s), 300)
omega_arr = odeint(domega_dt, Omega_0, t_arr_s, args=(A,)).ravel()
t_arr_gyr = t_arr_s / yr / 1e9

omega_skumanich = Omega_sun * (age_sun_gyr / t_arr_gyr) ** 0.5

fig, ax = plt.subplots(figsize=(9, 6))
ax.loglog(t_arr_gyr, omega_arr / Omega_sun, 'b-', lw=2,
          label='Numerical ODE (Weber-Davis)')
ax.loglog(t_arr_gyr, omega_skumanich / Omega_sun, 'r--', lw=2,
          label=r'Analytic $\Omega \propto t^{-1/2}$ (Skumanich)')
ages_data = [0.01, 0.1, 0.6, 4.6]
omega_data = [50, 10, 3, 1]  # in solar units
ax.plot(ages_data, omega_data, 'ks', ms=10, label='Open cluster medians (schematic)')
ax.set_xlabel('Age (Gyr)')
ax.set_ylabel(r'$\Omega_\star / \Omega_\odot$')
ax.set_title('Skumanich spin-down (Vidotto 2021 Eq. 68)')
ax.grid(alpha=0.3, which='both')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

## 5. Synthetic Lyman-alpha Absorption from Astrospheric H Wall / Astrosphere 수소 벽 Ly-alpha 흡수

**English.** The hydrogen wall between bow shock and astropause creates a column of hot neutral H (charge-exchange products) at velocity ~$+50$ km/s and temperature ~$10^4-10^5$ K. We model the Ly-alpha absorption as a Gaussian optical-depth profile sitting on the stellar emission line.

**한국어.** Bow shock과 astropause 사이 수소 벽은 charge-exchange 산물의 뜨거운 중성 수소 기둥을 형성, 속도 ~$+50$ km/s, 온도 ~$10^4-10^5$ K. Ly-alpha 흡수를 별의 방출선 위에 있는 가우시안 광학 두께 프로파일로 모델링.

In [ ]:
def gaussian(v, v0, sigma):
    """Return normalised Gaussian profile.

    Args:
        v: Velocity array in km/s.
        v0: Line centre in km/s.
        sigma: Gaussian width in km/s.

    Returns:
        Gaussian amplitude array.
    """
    return np.exp(-0.5 * ((v - v0) / sigma) ** 2)

v = np.linspace(-300, 300, 1000)  # km/s relative to Ly-alpha rest

emission = 1.0 * gaussian(v, 0, 80)
tau_ism = 2.0 * gaussian(v, 0, 12)
tau_hwall = 0.6 * gaussian(v, 50, 30)

observed = emission * np.exp(-tau_ism - tau_hwall)
observed_ism_only = emission * np.exp(-tau_ism)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(v, emission, 'k:', lw=1, label='Stellar Ly-alpha emission')
ax.plot(v, observed_ism_only, 'b--', lw=1.5, label='With ISM absorption only')
ax.plot(v, observed, 'r-', lw=2, label='With ISM + hydrogen wall absorption')
ax.axvspan(30, 90, color='orange', alpha=0.2, label='Hydrogen wall absorption region')
ax.set_xlabel('Velocity (km/s)')
ax.set_ylabel('Normalised flux')
ax.set_title('Synthetic Ly-alpha astrospheric absorption (Vidotto 2021, Sect. 2.1)')
ax.grid(alpha=0.3); ax.legend(loc='best'); ax.set_ylim(0, 1.1)
plt.tight_layout(); plt.show()

print('The red-wing absorption (+30 to +90 km/s) is the signature from which')
print('P_ram = Mdot * u_inf / (4 pi r^2) and hence Mdot can be inferred.')

## 6. Earth's Atmospheric Loss from the Young Solar Wind / 젊은 태양풍에 의한 지구 대기 손실

**English.** Using the energy-limited escape formula $\dot M_{atm} = \eta \pi R_P^3 F_{XUV}/(G M_P)$ and Ribas et al. (2005) XUV evolution, we integrate the cumulative atmospheric mass lost from Earth since 0.1 Gyr. Compare with Mars (no protective magnetosphere, hence higher $\eta$).

**한국어.** 에너지 한계 escape 공식 $\dot M_{atm} = \eta \pi R_P^3 F_{XUV}/(G M_P)$과 Ribas et al.(2005) XUV 진화를 사용하여, 0.1 Gyr부터 지구의 누적 대기 질량 손실을 적분. 화성(보호 자기권 없음, 높은 $\eta$)과 비교.

In [ ]:
R_earth = 6.371e8; M_earth = 5.972e27  # cm, g
R_mars = 3.390e8;  M_mars  = 6.417e26
F_XUV_sun_1au_now = 4.6  # erg/cm^2/s (present-day solar XUV at 1 au, Ribas 2005)

def F_XUV_at_age(age_gyr, F_now=F_XUV_sun_1au_now):
    """Return solar XUV flux at 1 au at a given stellar age.

    Args:
        age_gyr: Age in Gyr.
        F_now: Present-day (4.6 Gyr) XUV flux at 1 au in erg/cm^2/s.

    Returns:
        XUV flux in erg/cm^2/s.
    """
    return F_now * (age_gyr / age_sun_gyr) ** (-1.23)

def mdot_atm_energy_limited(F_XUV, R_P, M_P, eta=0.15):
    """Return energy-limited atmospheric mass-loss rate in g/s.

    Args:
        F_XUV: XUV flux at the planet in erg/cm^2/s.
        R_P: Planet radius in cm.
        M_P: Planet mass in g.
        eta: Heating efficiency, typically 0.1-0.3.

    Returns:
        Mass-loss rate in g/s.
    """
    return eta * np.pi * R_P ** 3 * F_XUV / (G * M_P)

ages_int = np.linspace(0.05, 4.6, 200)
F_XUV_arr = F_XUV_at_age(ages_int)
mdot_earth = mdot_atm_energy_limited(F_XUV_arr, R_earth, M_earth, eta=0.15)
mdot_mars = mdot_atm_energy_limited(F_XUV_arr, R_mars, M_mars, eta=0.30)

cum_earth = np.cumsum(mdot_earth[:-1] * np.diff(ages_int) * 1e9 * yr)
cum_mars = np.cumsum(mdot_mars[:-1] * np.diff(ages_int) * 1e9 * yr)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].loglog(ages_int, mdot_earth, 'b-', label='Earth (eta=0.15)')
ax[0].loglog(ages_int, mdot_mars, 'r-', label='Mars (eta=0.30, no B-field)')
ax[0].set_xlabel('Age (Gyr)'); ax[0].set_ylabel(r'$\dot M_{atm}$ (g/s)')
ax[0].set_title('Instantaneous atmospheric loss rate')
ax[0].grid(alpha=0.3); ax[0].legend()
ax[1].semilogx(ages_int[:-1], cum_earth / M_earth, 'b-', label='Earth')
ax[1].semilogx(ages_int[:-1], cum_mars / M_mars, 'r-', label='Mars')
ax[1].axhline(1e-4, color='gray', linestyle=':', label='Earth atm fraction today')
ax[1].set_xlabel('Age (Gyr)'); ax[1].set_ylabel('Cumulative lost mass / M_planet')
ax[1].set_title('Cumulative atmospheric mass lost')
ax[1].grid(alpha=0.3); ax[1].legend()
plt.tight_layout(); plt.show()

print(f'Integrated atm mass lost by Earth over 4.55 Gyr: {cum_earth[-1]:.2e} g')
print(f'  (Earth mass = {M_earth:.2e} g)')
print(f'Integrated atm mass lost by Mars over 4.55 Gyr: {cum_mars[-1]:.2e} g')
print(f'  (Mars mass = {M_mars:.2e} g)')

## 7. Empirical $\dot M - F_X$ Scaling (Vidotto 2021 Eq. 17) / 경험적 $\dot M - F_X$ scaling

**English.** Vidotto (2021) Eq. 17: $\dot M/R_\star^2 = 10^{-2.75}\,F_X^{0.66}\,M_\odot/\textrm{yr}/R_\odot^2$. Plot this along with schematic observation points (astrosphere / exoplanet / prominence).

**한국어.** Vidotto(2021) 식 17: 위 식. astrosphere / 외계행성 / prominence 관측 점(모식)과 함께 도시.

In [ ]:
fx_arr = np.logspace(3, 8, 100)
# Mdot/R^2 [in M_sun/yr/R_sun^2] = 10^-2.75 * F_X^0.66
mdot_r2 = 10**(-2.75) * fx_arr**0.66
mdot_r2_relative = mdot_r2 / Mdot_sun  # relative to solar

obs = [
    (1e4, 0.5, 'orange', 'Astrosphere (old stars)'),
    (1e5, 5, 'orange', None),
    (5e5, 30, 'orange', None),
    (1e6, 100, 'green', 'Prominence (young active)'),
    (1e7, 1000, 'green', None),
    (1e8, 4500, 'green', None),
    (1e4, 1, 'black', 'Sun (present)')]

fig, ax = plt.subplots(figsize=(9, 6))
ax.loglog(fx_arr, mdot_r2_relative, 'k-', lw=2,
          label=r'Vidotto Eq. 17: $\dot M/R_\star^2 \propto F_X^{0.66}$')
for fx, m, c, lab in obs:
    ax.plot(fx, m, 'o', color=c, ms=10, label=lab)
ax.set_xlabel(r'$F_X$ (erg cm$^{-2}$ s$^{-1}$)')
ax.set_ylabel(r'$(\dot M/R_\star^2)/(\dot M_\odot/R_\odot^2)$')
ax.set_title('Empirical mass-loss vs X-ray flux (Vidotto 2021, Fig. 8)')
ax.grid(alpha=0.3, which='both')
handles, labels = ax.get_legend_handles_labels()
unique = [(h, l) for i, (h, l) in enumerate(zip(handles, labels)) if l and l not in labels[:i]]
ax.legend(*zip(*unique), loc='best')
plt.tight_layout(); plt.show()

## 8. Heliosphere Evolution / 태양권 진화

**English.** Using Eq. 74 of Vidotto (2021), compute how the heliospheric (astrospheric) radius evolved alongside $\dot M$:
$$r_{astro} = \left[\frac{\dot M u_\infty}{4\pi \rho_{ISM}u_{ISM}^2}\right]^{1/2}$$
At 600 Myr, heliosphere was 1300-1700 au (vs 122 au today).

**한국어.** Vidotto(2021) 식 74를 사용해 heliosphere(astrosphere) 반경이 $\dot M$과 함께 어떻게 진화했는지 계산. 600 Myr에는 1300-1700 au(현재 122 au).

In [ ]:
rho_ISM = 2.0e-25   # g/cm^3 (typical local ISM)
u_ISM = 2.5e6       # cm/s (~25 km/s relative motion)

def r_astro(mdot_ratio, u_inf=u_inf_sun):
    """Return astrospheric radius in au.

    Args:
        mdot_ratio: Mass-loss rate in units of solar.
        u_inf: Terminal wind speed in cm/s.

    Returns:
        Astrospheric radius in au.
    """
    mdot_cgs = mdot_ratio * Mdot_sun * M_sun / yr
    r_cm = np.sqrt(mdot_cgs * u_inf / (4 * np.pi * rho_ISM * u_ISM**2))
    return r_cm / au

mdot_ratio_arr = mdot_vidotto(ages_int, -0.99)
r_astro_arr = r_astro(mdot_ratio_arr)

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(ages_int, r_astro_arr, 'b-', lw=2)
ax.axhline(122, color='gray', linestyle=':', label='Today: ~122 au')
ax.axvline(0.6, color='red', linestyle=':', alpha=0.5)
ax.axvline(age_sun_gyr, color='orange', linestyle=':', alpha=0.5)
ax.set_xlabel('Age (Gyr)'); ax.set_ylabel('Heliospheric radius (au)')
ax.set_title('Heliospheric size evolution (Vidotto 2021, Eq. 74)')
ax.grid(alpha=0.3, which='both'); ax.legend()
plt.tight_layout(); plt.show()

print(f'Heliosphere at 0.6 Gyr: r ~ {r_astro(mdot_vidotto(0.6, -0.99)):.0f} au (Rodgers-Lee 2020 says 1300-1700 au)')
print(f'Heliosphere today:      r ~ {r_astro(1.0):.0f} au (observed ~122 au)')

## 9. Summary / 요약

**English.** This notebook reproduced the quantitative core of Vidotto (2021):

1. **Parker winds**: transonic isothermal solutions pass through $r_c = GM/(2c_s^2)$; $T$ controls $r_c$ and $u_\infty$.
2. **Evolution**: $\dot M \propto t^{-0.99}$ vs $t^{-2.33}$ — young Sun had 10-1000× today's $\dot M$.
3. **Angular momentum**: $\dot J = \tfrac{2}{3}\dot M\Omega r_A^2 \simeq 10^{30}$ erg for the present Sun, consistent with observation.
4. **Skumanich**: numerical integration of $\dot\Omega \propto -\Omega^3$ reproduces $t^{-1/2}$ spindown.
5. **Ly-alpha astrosphere**: red-wing H-wall absorption is the signature used to infer $\dot M$.
6. **Planetary atmospheres**: young Sun's XUV drives substantial atmospheric loss on Earth/Mars.
7. **Fx scaling**: Eq. 17 unifies detection methods into a single power-law.
8. **Heliosphere**: was ~10× larger at 600 Myr, suppressing GCRs at Earth.

**한국어.** 이 노트북은 Vidotto(2021)의 정량적 핵심을 재현했다:

1. **Parker 항성풍**: transonic 등온 해는 $r_c = GM/(2c_s^2)$을 지나며 $T$가 $r_c$·$u_\infty$를 결정.
2. **진화**: $\dot M \propto t^{-0.99}$ 대 $t^{-2.33}$ — 젊은 태양은 현재의 10-1000배 $\dot M$.
3. **각운동량**: 현재 태양의 $\dot J = \tfrac{2}{3}\dot M\Omega r_A^2 \simeq 10^{30}$ erg, 관측과 일치.
4. **Skumanich**: $\dot\Omega \propto -\Omega^3$의 수치 적분이 $t^{-1/2}$ 감속 재현.
5. **Ly-alpha astrosphere**: 적색 날개 H-벽 흡수가 $\dot M$ 추론 서명.
6. **행성 대기**: 젊은 태양 XUV가 지구/화성 대기의 큰 손실을 유도.
7. **Fx scaling**: 식 17이 탐지 방법들을 단일 멱법칙으로 통합.
8. **태양권**: 600 Myr에 ~10배 컸으며, 지구 GCR 억제.